# Baseline Retention Classification Model

This notebook builds and evaluates the first machine-learning model for employee attrition prediction.

The modeling design uses:

- Snapshot date: June 30, 2025
- Prediction window: July 1, 2025 through June 30, 2026
- Target: `attrition_next_12m`

The baseline workflow includes:

1. Feature and target separation
2. Stratified train/test split
3. Missing-value imputation
4. Numerical feature scaling
5. Categorical one-hot encoding
6. Dummy Classifier baseline
7. Logistic Regression classifier
8. Classification-metric evaluation

The purpose of this stage is to establish a reproducible baseline before testing more advanced models.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "retention_modeling_dataset.csv"
)


retention = pd.read_csv(
    DATA_PATH
)


print(
    "Dataset shape:",
    retention.shape,
)

Dataset shape: (7386, 39)


## 1. Attrition target

In [2]:
target_summary = (
    retention[
        "attrition_next_12m"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "attrition_next_12m"
    )
    .reset_index(
        name="employee_count"
    )
)

target_summary

,attrition_next_12m,employee_count
0,0,6759
1,1,627


In [3]:
attrition_rate = (
    100
    * retention[
        "attrition_next_12m"
    ].mean()
)

print(
    f"Attrition rate: "
    f"{attrition_rate:.2f}%"
)

Attrition rate: 8.49%


In [4]:
TARGET_COLUMN = (
    "attrition_next_12m"
)


NUMERICAL_FEATURES = [
    "approx_age",
    "tenure_years",
    "years_experience_at_hire",
    "initial_base_salary",
    "base_salary",
    "bonus_target",
    "equity_value",
    "salary_growth_percent",
    "days_since_compensation_change",
    "compensation_record_count",
    "promotion_compensation_count",
    "performance_rating",
    "goal_completion",
    "days_since_review",
    "review_count",
    "average_performance_rating",
    "completed_training_programs",
    "failed_training_programs",
    "in_progress_training_programs",
    "completed_training_hours",
    "average_training_score",
    "prior_promotion_events",
    "prior_transfer_events",
    "prior_manager_change_events",
    "prior_leave_events",
    "prior_change_events",
    "days_since_last_change_event",
]


CATEGORICAL_FEATURES = [
    "employment_type",
    "education_level",
    "application_source",
    "hire_department_name",
    "hire_region",
    "hire_job_family",
    "hire_job_level",
    "promotion_recommended",
]


FEATURE_COLUMNS = (
    NUMERICAL_FEATURES
    + CATEGORICAL_FEATURES
)


X = retention[
    FEATURE_COLUMNS
].copy()


y = (
    retention[
        TARGET_COLUMN
    ]
    .astype(int)
    .copy()
)


print(
    "Feature matrix:",
    X.shape,
)

print(
    "Target:",
    y.shape,
)

Feature matrix: (7386, 35)
Target: (7386,)


In [5]:
from sklearn.model_selection import (
    train_test_split
)


X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )
)


print(
    "Training rows:",
    len(X_train),
)

print(
    "Testing rows:",
    len(X_test),
)

Training rows: 5908
Testing rows: 1478


In [6]:
split_summary = pd.DataFrame(
    {
        "dataset": [
            "Full",
            "Training",
            "Testing",
        ],

        "rows": [
            len(y),
            len(y_train),
            len(y_test),
        ],

        "attrition_rate": [
            y.mean(),
            y_train.mean(),
            y_test.mean(),
        ],
    }
)


split_summary[
    "attrition_rate_percent"
] = (
    100
    * split_summary[
        "attrition_rate"
    ]
)


split_summary

,dataset,rows,attrition_rate,attrition_rate_percent
0,Full,7386,0.084890,8.489033
1,Training,5908,0.084970,8.496953
2,Testing,1478,0.084574,8.457375


In [7]:
from sklearn.compose import (
    ColumnTransformer
)

from sklearn.impute import (
    SimpleImputer
)

from sklearn.pipeline import (
    Pipeline
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),

        (
            "scaler",
            StandardScaler(),
        ),
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),

        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            NUMERICAL_FEATURES,
        ),

        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
    ]
)

## 2. Dummy Classifier baseline

In [8]:
from sklearn.dummy import (
    DummyClassifier
)


dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42,
)


dummy_model.fit(
    X_train,
    y_train,
)


dummy_predictions = (
    dummy_model.predict(
        X_test
    )
)


dummy_probabilities = (
    dummy_model.predict_proba(
        X_test
    )[:, 1]
)

## 3. Logistic Regression baseline

In [9]:
from sklearn.linear_model import (
    LogisticRegression
)


logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)


logistic_model.fit(
    X_train,
    y_train,
)


logistic_predictions = (
    logistic_model.predict(
        X_test
    )
)


logistic_probabilities = (
    logistic_model.predict_proba(
        X_test
    )[:, 1]
)

In [10]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def calculate_metrics(
    model_name,
    predictions,
    probabilities,
):

    return {
        "model": model_name,

        "accuracy": accuracy_score(
            y_test,
            predictions,
        ),

        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            y_test,
            predictions,
            zero_division=0,
        ),

        "roc_auc": roc_auc_score(
            y_test,
            probabilities,
        ),

        "pr_auc": average_precision_score(
            y_test,
            probabilities,
        ),
    }


model_comparison = pd.DataFrame(
    [
        calculate_metrics(
            "Dummy Classifier",
            dummy_predictions,
            dummy_probabilities,
        ),

        calculate_metrics(
            "Logistic Regression",
            logistic_predictions,
            logistic_probabilities,
        ),
    ]
)


model_comparison.round(4)

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Dummy Classifier,0.9154,0.0000,0.000,0.0000,0.5000,0.0846
1,Logistic Regression,0.5981,0.1376,0.712,0.2306,0.6968,0.1709


In [11]:
from sklearn.metrics import (
    confusion_matrix
)


confusion = confusion_matrix(
    y_test,
    logistic_predictions,
)


confusion_table = pd.DataFrame(
    confusion,
    index=[
        "Actual Retained",
        "Actual Attrition",
    ],
    columns=[
        "Predicted Retained",
        "Predicted Attrition",
    ],
)


confusion_table

,Predicted Retained,Predicted Attrition
Actual Retained,795,558
Actual Attrition,36,89


In [12]:
test_predictions = pd.DataFrame(
    {
        "employee_id": (
            retention.loc[
                X_test.index,
                "employee_id",
            ]
        ),

        "actual_attrition": (
            y_test
        ),

        "predicted_attrition": (
            logistic_predictions
        ),

        "attrition_probability": (
            logistic_probabilities
        ),
    }
)


test_predictions = (
    test_predictions
    .sort_values(
        "attrition_probability",
        ascending=False,
    )
)


test_predictions.head(20)

,employee_id,actual_attrition,predicted_attrition,attrition_probability
2747,103686,1,1,0.901905
922,101205,0,1,0.842157
969,101264,0,1,0.840639
2833,103814,1,1,0.830257
4012,105440,1,1,0.825593
7145,109657,0,1,0.807670
1777,102291,0,1,0.807324
1760,102264,0,1,0.805777
2931,103957,0,1,0.803628
5720,107759,0,1,0.800371


## 4. Baseline model validation

In [13]:
baseline_checks = pd.Series(
    {
        "training set is not empty": (
            len(X_train) > 0
        ),

        "testing set is not empty": (
            len(X_test) > 0
        ),

        "training contains both classes": (
            y_train.nunique() == 2
        ),

        "testing contains both classes": (
            y_test.nunique() == 2
        ),

        "prediction count matches test set": (
            len(
                logistic_predictions
            )
            == len(y_test)
        ),

        "probability count matches test set": (
            len(
                logistic_probabilities
            )
            == len(y_test)
        ),

        "probabilities are between zero and one": (
            (
                logistic_probabilities
                >= 0
            ).all()

            and

            (
                logistic_probabilities
                <= 1
            ).all()
        ),

        "logistic ROC-AUC is valid": (
            0
            <= roc_auc_score(
                y_test,
                logistic_probabilities,
            )
            <= 1
        ),
    },
    name="passed",
)


baseline_checks

training set is not empty                 True
testing set is not empty                  True
training contains both classes            True
testing contains both classes             True
prediction count matches test set         True
probability count matches test set        True
probabilities are between zero and one    True
logistic ROC-AUC is valid                 True
Name: passed, dtype: bool

In [14]:
if baseline_checks.all():

    print(
        "All baseline model "
        "validation checks passed."
    )

else:

    print(
        "One or more baseline model "
        "validation checks failed."
    )

All baseline model validation checks passed.


## 5. Conclusions

The baseline retention-modeling pipeline successfully trains and evaluates an attrition classifier.

### Data preparation

- Identifier and date-control columns are excluded from model features.
- Numerical missing values are imputed using the median.
- Numerical features are standardized.
- Categorical missing values are imputed using the most frequent category.
- Categorical features are one-hot encoded.

### Model evaluation

A Dummy Classifier provides a simple benchmark.

Logistic Regression provides the first feature-based attrition model.

The models are evaluated using:

- Accuracy
- Precision
- Recall
- F1 score
- ROC-AUC
- PR-AUC
- Confusion matrix

Because attrition may be an imbalanced classification problem, accuracy alone is not sufficient for evaluating model usefulness.

### Baseline limitations

- The current model uses one historical snapshot.
- Logistic Regression assumes a relatively simple relationship between transformed features and attrition risk.
- The default classification threshold is 0.50.
- Model probabilities have not been calibrated.
- Hyperparameters have not been tuned.
- More complex classification models have not yet been compared.

### Next step

The next stage will compare multiple classification models and evaluate whether tree-based methods improve attrition prediction over the Logistic Regression baseline.